# Sweep the spectra while following every hybridisation term
For a 2×2 pairs system at g/2π = 5, 20 and 40 kHz, shows the homodyne and heterodyne spectra, the x₁ PSD and the x₁x₂ CSD (LAM, and Power Series H truncated at orders 3, 2 and 1) with a red line marking the sweep frequency, above the hybridisation functions R_jk and the sieved determinant terms of orders 2, 3 and 4 in the complex plane.

Runtime: about 2-3 min by default, which shows the last frame of each sweep as a static figure. Set `MAKE_ANIMATIONS = True` to write the three animations to `output/` (about 30 min).

# START

# Sweep of all spectra while visualising all hybridisation terms

In [ ]:
from pathlib import Path
import matplotlib
import matplotlib.animation
from IPython.display import display
import numpy as np
import matplotlib.pyplot as plt
from math import factorial
from fractions import Fraction
from levisim import *

set_plt_rcParams_to_xmgrace_style()

# --- run settings ---------------------------------------------------------------------------------
# False (default): every animation is stepped through its frames without being rendered and shown as
# one static figure of its last frame (with every trail), which takes seconds.
# True: every animation is also written to output/ (GIF, or MP4 when ffmpeg is installed), which
# takes minutes. output/ is not tracked by git.
MAKE_ANIMATIONS = False
# keep every n-th frame; n > 1 gives a quick test of the full run
FRAME_STRIDE = 1
# resolution of the written animation files
ANIMATION_DPI = 75
OUTPUT_DIR = Path('output')


class SweepAnimation:
    """
    A frequency-sweep animation, update(frame) over num_frames frames, rendered only on request

    show(name) writes it to OUTPUT_DIR when MAKE_ANIMATIONS is True, otherwise it just runs
    update over the frames; either way it then displays the figure, which holds the last frame.
    """
    def __init__(self, fig, update, num_frames, fps=20):
        self.fig, self.update, self.fps = fig, update, fps
        self.frames = list(range(0, num_frames, FRAME_STRIDE))
        # closed so that the notebook does not also show the empty figure when the cell ends
        plt.close(fig)

    def show(self, name):
        if MAKE_ANIMATIONS:
            OUTPUT_DIR.mkdir(exist_ok=True)
            writer = 'ffmpeg' if matplotlib.animation.writers.is_available('ffmpeg') else 'pillow'
            path = OUTPUT_DIR / f'{name}.{"mp4" if writer == "ffmpeg" else "gif"}'
            animation = matplotlib.animation.FuncAnimation(self.fig, self.update, frames=self.frames, blit=False)
            animation.save(path, writer=writer, fps=self.fps, dpi=ANIMATION_DPI)
            print(f'wrote {path} ({len(self.frames)} frames)')
        else:
            for frame in self.frames:
                self.update(frame)
        self.fig.canvas.draw()
        display(self.fig)


# https://oeis.org/A000178
def superfactorial(n):
    result = 1
    for digit in range(n + 1):
        result *= factorial(digit)
    return round(result)

def nPk(n, k):
    return round(factorial(n) / factorial(n - k))

# https://oeis.org/A279046
def En_numerator(n):
    numerator_terms = [(-1)**j * superfactorial(n) / factorial(j) for j in range(n + 1)]
    result = sum(numerator_terms)
    return round(result)

def En_denominator(n):
    return round(superfactorial(n))

# final numerator = https://oeis.org/A053557
# final denominator = https://oeis.org/A053556
def En(n):
    return Fraction(
        numerator=En_numerator(n),
        denominator=En_denominator(n),
    )

def generate_Rjk_sweep_animation(Ω_model_type, g=-20*kHz):
    """
    Generate a `FuncAnimation` that visualises hybridisation term values over frequency sweep of heterodyne spectrum
    """
    gas_kwargs = dict(P=1e-6*mbar, T=300*K, vth=500*metre/second)
    cavity_kwargs = dict(Δ=-450*kHz, κ=400*kHz, θhom=π/4)
    toy_params = dict(Ω_model=Ω_model_type, toy_gx=g, toy_gy=g)

    system = '2×2'
    Nd = system.split('×')
    N, d = int(Nd[0]), int(Nd[1])
    D = N * d

    ω_range = np.arange(250, 350 + 0.1, 0.1) * kHz
    mtrap = MultiTrap(
        num_traps=N,
        Cartesian_axes_per_trap=d,
        ω_range=ω_range,
        gas_kwargs=gas_kwargs,
        cavity_kwargs=cavity_kwargs,
        toy_params=toy_params,
        generate_power_series=True,
    )

    mosaic = [
        [SpectrumType.HOMODYNE] * 4,
        [SpectrumType.HETERODYNE] * 4,
        [SpectrumType.POSITION_PSD] * 4,
        [SpectrumType.POSITION_CSD] * 4,
        ['Rjk', 'Rjk', 'Rjk_2', 'Rjk_2'],
        ['Rjk_3', 'Rjk_3', 'Rjk_4', 'Rjk_4'],
    ]
    mosaic = [
        [SpectrumType.HOMODYNE] * 4,
        [SpectrumType.HETERODYNE] * 4,
        [SpectrumType.POSITION_PSD] * 4,
        [SpectrumType.POSITION_CSD] * 4,
        ['Rjk'] * 4,
        ['Rjk'] * 4,
        ['Rjk_2'] * 4,
        ['Rjk_2'] * 4,
        ['Rjk_3'] * 4,
        ['Rjk_3'] * 4,
        ['Rjk_4'] * 4,
        ['Rjk_4'] * 4,
    ]
    fig = plt.figure(figsize=(1.5 * LATEX_FIG_WIDTH, 3.5 * len(mosaic)), dpi=100)

    ax_dict = fig.subplot_mosaic(mosaic)

    hom, het = mtrap.homodyne_and_heterodyne_spectra(QuantumLinearTheoryMethod.LINEAR_AMPLIFIER_MODEL)
    x1_PSD = mtrap.position_power_spectrum('x1', QuantumLinearTheoryMethod.LINEAR_AMPLIFIER_MODEL)
    x1_x2_CSD = mtrap.position_cross_spectrum('x1', 'x2', QuantumLinearTheoryMethod.LINEAR_AMPLIFIER_MODEL)

    mtrap.evaluation_order = 3
    hom_3, het_3 = mtrap.homodyne_and_heterodyne_spectra(QuantumLinearTheoryMethod.POWER_SERIES_HYBRIDISATION)
    x1_PSD_3 = mtrap.position_power_spectrum('x1', QuantumLinearTheoryMethod.POWER_SERIES_HYBRIDISATION)
    x1_x2_CSD_3 = mtrap.position_cross_spectrum('x1', 'x2', QuantumLinearTheoryMethod.POWER_SERIES_HYBRIDISATION)

    mtrap.evaluation_order = 2
    hom_2, het_2 = mtrap.homodyne_and_heterodyne_spectra(QuantumLinearTheoryMethod.POWER_SERIES_HYBRIDISATION)
    x1_PSD_2 = mtrap.position_power_spectrum('x1', QuantumLinearTheoryMethod.POWER_SERIES_HYBRIDISATION)
    x1_x2_CSD_2 = mtrap.position_cross_spectrum('x1', 'x2', QuantumLinearTheoryMethod.POWER_SERIES_HYBRIDISATION)

    mtrap.evaluation_order = 1
    hom_1, het_1 = mtrap.homodyne_and_heterodyne_spectra(QuantumLinearTheoryMethod.POWER_SERIES_HYBRIDISATION)
    x1_PSD_1 = mtrap.position_power_spectrum('x1', QuantumLinearTheoryMethod.POWER_SERIES_HYBRIDISATION)
    x1_x2_CSD_1 = mtrap.position_cross_spectrum('x1', 'x2', QuantumLinearTheoryMethod.POWER_SERIES_HYBRIDISATION)

    homs = [hom, hom_3, hom_2, hom_1]
    hets = [het, het_3, het_2, het_1]
    x1_PSDs = [x1_PSD, x1_PSD_3, x1_PSD_2, x1_PSD_1]
    x1_x2_CSDs = [x1_x2_CSD, x1_x2_CSD_3, x1_x2_CSD_2, x1_x2_CSD_1]

    red_vlines = {}
    for spectrum_type in [
        SpectrumType.HOMODYNE,
        SpectrumType.HETERODYNE,
        SpectrumType.POSITION_PSD,
        SpectrumType.POSITION_CSD,
    ]:
        match spectrum_type:
            case SpectrumType.HOMODYNE:
                label = 'Homodyne'
                ylabel = '[photons/Hz]'
                # first and last colours too dark / bright
                colours = discrete_viridis[D + 2]
                vals_list = homs
            case SpectrumType.HETERODYNE:
                label = 'Heterodyne'
                ylabel = '[photons/Hz]'
                colours = discrete_viridis[D + 2]
                vals_list = hets
            case SpectrumType.POSITION_PSD:
                label = r'$x_1$ PSD'
                ylabel = '[phonons/Hz]'
                colours = discrete_plasma[D + 2]
                vals_list = x1_PSDs
            case SpectrumType.POSITION_CSD:
                label = r'$x_1, x_2$ CSD'
                ylabel = '[phonons/Hz]'
                colours = discrete_plasma[D + 2]
                vals_list = x1_x2_CSDs

        ax = ax_dict[spectrum_type]
        ax.plot(ω_range / kHz, vals_list[0], label=label, color=colours[1], lw=4, alpha=0.5)
        ax.plot(ω_range / kHz, vals_list[1], label='(3rd order)', color=colours[2], lw=3, alpha=0.5)
        ax.plot(ω_range / kHz, vals_list[2], label='(2nd order)', color=colours[3], lw=2, alpha=0.5)
        ax.plot(ω_range / kHz, vals_list[3], label='(1st order)', color=colours[4], lw=1, alpha=0.5)

        red_vlines[spectrum_type] = ax.axvline(x=ω_range[0] / kHz, color=RGB_red)

        ax.set_xlabel('Frequency [kHz]')
        ax.set_ylabel(ylabel)
        ax.set_xlim(ω_range.min() / kHz, ω_range.max() / kHz)
        if spectrum_type != SpectrumType.POSITION_CSD:
            ax.set_yscale('log')
        ax.legend(loc='center right', prop=dict(size=8))
        ax.grid(alpha=0.1)

    all_arrows = {}
    all_output_trails = {}
    all_trails = {}
    all_terms = {}
    for R_key in [
        'Rjk',
        'Rjk_2',
        'Rjk_3',
        'Rjk_4',
    ]:
        match R_key:
            case 'Rjk':
                terms = mtrap.all_Rjk
                assert len(terms) == D**2 - D
            case 'Rjk_2':
                terms = sieve_power_series(mtrap.deranged_determinant_by_order[2], 2)
                n, a = D, 2
                assert len(terms) == round(nPk(n, a) * En(a))
            case 'Rjk_3':
                terms = sieve_power_series(mtrap.deranged_determinant_by_order[3], 3)
                n, a = D, 3
                assert len(terms) == round(nPk(n, a) * En(a))
            case 'Rjk_4':
                terms = sieve_power_series(mtrap.deranged_determinant_by_order[4], 4)
                n, a = D, 4
                assert len(terms) == round(nPk(n, a) * En(a))

        ax = ax_dict[R_key]
        ax.set_aspect('equal')
        xlims, ylims = get_latex_doc_lims(1, 1e0)
        ax.set_xlim(*xlims)
        ax.set_ylim(*ylims)
        ax.axhline(0, color=black, linewidth=0.5)
        ax.axvline(0, color=black, linewidth=0.5)
        ax.grid(alpha=0.1)

        circle = matplotlib.patches.Circle((0, 0), radius=1, color='k', alpha=0.05)
        ax.add_patch(circle)

        colours = colours_50
        arrows = [
            ax.plot([], [], color=colour, marker='o')
            for colour in colours
        ]
        arrows = sum(arrows, [])
        output_trails = [
            ax.scatter([], [], color=colour, alpha=0.5, s=10)
            for colour in colours
        ]
        trails = [[] for colour in colours]

        num_terms = len(terms)
        all_arrows[R_key] = arrows[:num_terms]
        all_output_trails[R_key] = output_trails[:num_terms]
        all_trails[R_key] = trails[:num_terms]
        all_terms[R_key] = terms[:num_terms]

    # NOTE: the couplings are negative, so compare |g| (a plain `g / kHz <= 10` was true for every g)
    if abs(g) / kHz <= 10:
        ω_input = np.linspace(280, 305, 250 + 1, endpoint=True) * kHz
    elif abs(g) / kHz <= 20:
        ω_input = np.linspace(275, 305, 300 + 1, endpoint=True) * kHz
    else:
        ω_input = np.linspace(265, 305, 400 + 1, endpoint=True) * kHz

    strong_hybridisation_region = []
    strong_hybridisation_values = []
    for ω in ω_input:
        for ii, term in enumerate(all_terms['Rjk']):
            fω = mtrap.hybridisation_term_value(term, ω)
            if np.abs(fω) >= 1:
                strong_hybridisation_region.append(ω)
                strong_hybridisation_values.append(np.abs(fω))
                # print(f'Abs[{term_string(term)[1:]}(ω/2π = {ω / kHz:.1f} kHz)] = {np.abs(fω):.2f} > 1')

    print(f'{min(strong_hybridisation_region) / kHz=}')
    print(f'{max(strong_hybridisation_region) / kHz=}')
    strong_hybridisation_range = max(strong_hybridisation_region) - min(strong_hybridisation_region)
    print(f'{strong_hybridisation_range / kHz=:.3f}')
    strongest_hybridisation = max(strong_hybridisation_values)
    print(f'{strongest_hybridisation=:.3f}')

    def update(frame):
        ω = ω_input[frame]
        for R_key, terms in all_terms.items():
            for ii, term in enumerate(terms):
                fω = mtrap.hybridisation_term_value(term, ω)

                all_arrows[R_key][ii].set_label(rf'${term_latex(term, with_ω=False)} = ${fω:.2f}')
                all_arrows[R_key][ii].set_data([0, fω.real], [0, fω.imag])
                all_trails[R_key][ii].append((fω.real, fω.imag))
                all_output_trails[R_key][ii].set_offsets(all_trails[R_key][ii])

        for ax_key, ax in ax_dict.items():
            if 'Rjk' in str(ax_key):
                ax.legend(bbox_to_anchor=(1, 0), loc='upper right', prop=dict(size=7), ncol=2)
                ax.set_title(rf'$\omega/2\pi=${ω / kHz:.2f} kHz')

        for spectrum_type in [
            SpectrumType.HOMODYNE,
            SpectrumType.HETERODYNE,
            SpectrumType.POSITION_PSD,
            SpectrumType.POSITION_CSD,
        ]:
            red_vlines[spectrum_type].set_xdata([ω / kHz, ω / kHz])
            ax = ax_dict[spectrum_type]
            ax.set_title(rf'Red line at $\omega/2\pi=${ω / kHz:.2f} kHz' + '\n' + rf'$g/2\pi = ${g / kHz:.0f} kHz')

        aa = list(_ for _ in all_arrows.values())
        all_arrows_list = aa[0] + aa[1] + aa[2] + aa[3]

        tt = list(_ for _ in all_output_trails.values())
        all_output_trails_list = tt[0] + tt[1] + tt[2] + tt[3]

        return (*all_arrows_list, *all_output_trails_list)

    ani = SweepAnimation(fig, update, len(ω_input))
    save_name = f'{D}D_{N}x{d}_spectra_and_hybridisation_terms_g={abs(g) / kHz:.0f}kHz'
    return save_name, ani

name_5, ani_5 = generate_Rjk_sweep_animation('pairs', g=-5*kHz)
name_20, ani_20 = generate_Rjk_sweep_animation('pairs', g=-20*kHz)
name_40, ani_40 = generate_Rjk_sweep_animation('pairs', g=-40*kHz)

In [ ]:
ani_5.show(name_5)

In [ ]:
ani_20.show(name_20)

In [ ]:
ani_40.show(name_40)

# END